# 13 · Modelos fundacionales y la escalera de aprendizaje profundo

**Qué pregunta responde.** ¿Aporta el aprendizaje profundo —modelos fundacionales de series y adaptación— una señal mejor que el especialista tabular en esta microestructura? ¿Y escala con más datos?

**Qué entradas usa.** Resultados agregados publicados en `results/`: `fundacionales_escalera.csv`, `fundacionales_escalado.csv`, `fundacionales_n1_calibracion_terminal.csv`, `fundacionales_n3_ensemble_healthy.csv`. No requiere el corpus privado ni GPU; reproduce la lectura de las tablas y figuras de la memoria (cap. 3-5).

**Qué produce.** Las cuatro piezas del bloque fundacional: la escalera zero-shot → fine-tuning frente a los baselines clásicos; la curva de escalado tabular vs profundo; la calibración terminal (N1); y el diagnóstico del fallo fuera de muestra por *deep ensemble* (N3).

**Conclusión (2 frases).** El aprendizaje profundo, adaptado o no, **no supera** a los baselines clásicos ni al especialista tabular en esta tarea, y la curva de escalado muestra que **más datos no cierran la brecha**: el cuello de botella es el régimen de datos, no la capacidad del modelo. Las piezas exploratorias (N1, N3) explican *por qué*: el precio ya está calibrado y su informatividad se concentra en el tramo terminal, y el fallo fuera de muestra de la cabeza de salud es *concept drift* por cambio de régimen, no varianza.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent
RES = ROOT / 'results'
FIG = ROOT / 'reports' / 'memoria' / 'figures'

escalera  = pd.read_csv(RES / 'fundacionales_escalera.csv')
escalado  = pd.read_csv(RES / 'fundacionales_escalado.csv')
n1        = pd.read_csv(RES / 'fundacionales_n1_calibracion_terminal.csv')
n3        = pd.read_csv(RES / 'fundacionales_n3_ensemble_healthy.csv')
print('cargado:', escalera.shape, escalado.shape, n1.shape, n3.shape)


## 1. La escalera: fundacionales vs baselines clásicos

Todos los modelos predicen la dirección del tramo H60 sobre los **906 casos congelados** (foto apertura−45 s), con la misma capa económica (comisión taker oficial). MOMENT se evalúa en tres regímenes de adaptación crecientes: *zero-shot*, *linear-probing* y *fine-tuning* completo (en GPU).

La adaptación **ayuda de forma monótona** (45,5 % → 46,6 % → 47,3 %), pero ni el fine-tuning completo supera a `naive-drift` (53,1 %) ni a `ARIMA` (56,5 %). El fundacional, la herramienta más sofisticada, no es la más precisa aquí.


In [ ]:
escalera.sort_values('acierto_pct', ascending=False).reset_index(drop=True)


## 2. Curva de escalado: ¿escala mejor el aprendizaje profundo?

Se enfrentan la vía profunda (MOMENT *fine-tuned*) y la tabular (*gradient boosting* sobre características clásicas) sobre la **misma** tarea, ventanas y evaluación, entrenando sobre subconjuntos crecientes del corpus.

La vía **tabular domina a la profunda en todo el rango** (48–50 % vs 44–47 %) y la brecha no se cierra al añadir datos: responde en negativo la objeción «¿no ganaría el *deep* con más datos?». Con dos semanas de corpus, la capacidad de una red profunda es desventaja, no ventaja.

![Curva de escalado tabular vs profundo](../reports/memoria/figures/fig_scaling_tab_vs_deep.png)


In [ ]:
escalado.pivot(index='n_train', columns='via', values='acierto_pct')


## 3. N1 · ¿Cuándo el precio deja de ser informativo? (calibración terminal)

Tratando el precio medio como estimador de P(settle=1), se mide su calibración a lo largo de la vida del mercado con *proper scoring* (Brier) por tiempo a expiración — sin entrenar ningún modelo.

El precio es casi una **moneda al aire en la apertura** (Brier ≈ 0,23 ≈ 0,25, acierto 58 %) y solo se vuelve informativo en los **últimos ~15–20 min** (Brier < 0,10, acierto ≥ 90 %). Está bien calibrado en todo momento, pero su masa se polariza tarde: la informatividad se concentra en el tramo terminal, lo que convierte el aplanado de inventario en la palanca de riesgo dominante del maker.

![Calibración terminal: Brier y fiabilidad](../reports/memoria/figures/fig_n1_calibracion_terminal.png)


In [ ]:
n1[['tte_lo','tte_hi','n','brier','acc_dir_pct']]


## 4. N3 · ¿Por qué falla la cabeza de salud fuera de muestra? (deep ensemble)

Diez réplicas de la cabeza de salud entrenadas con semillas distintas descomponen la incertidumbre del colapso fuera de muestra (AUC train 0,79 → test 0,54).

El colapso es **estable entre semillas** (±0,005), el **desacuerdo epistémico no sube** fuera de muestra (≈0,03 en train y test) y **ensemblar no ayuda** (0,544 → 0,547): no es un problema de varianza. Un clasificador de dominio separa mayo de junio con **AUC 0,90** (liderado por la base del perpetuo): el fallo es *concept drift* por cambio de régimen, no ruido ni capacidad. Consecuencia: reentrenar con más datos del mismo periodo tiene techo.

![Diagnóstico del fallo OOS por deep ensemble](../reports/memoria/figures/fig_n3_ensemble_healthy.png)


In [ ]:
n3


## 5. Lectura conjunta

Las cuatro piezas convergen en la tesis del trabajo: **la complejidad del modelo no es el cuello de botella**, así que no compra rendimiento. El fundacional no bate al tabular; más datos no cambian el veredicto; el precio ya está calibrado y su señal es tardía; y el fallo fuera de muestra es de régimen, no de modelo. La consecuencia de diseño —el giro a la vía maker y el control de inventario en el tramo terminal— se sostiene sobre esta evidencia, no sobre una intuición.

*Reproducibilidad:* las cifras provienen de scripts `asignatura_*` (repo `edgerunner`) sobre el corpus privado; aquí se cargan sus salidas agregadas y anonimizadas desde `results/`. Semillas fijadas en los scripts originales.
